# Correct LangSmith Tool-Use Error Analysis

This notebook uses the real LangSmith thread export for the multi-agent TravelPlanner system:

- `thread_analysis/travel_agent/langsmith_runs.csv`: real `run_type=tool` runs from LangSmith
- `thread_analysis/travel_agent/tool_calls.csv`: one row per real tool run
- `thread_analysis/travel_agent/manifest.csv`: local run id, thread id, query, validation outcome
- `travel_agent/run-*.json`: final structured `TravelPlan` artifacts
- `baseline/run-*.json`: baseline Tavily `tool_calls` embedded in messages

The goal is to produce paper-ready error analysis for:

- repeated tool calls
- repeated queries
- dead loops
- timeouts / long-running tool calls
- cascading errors
- structural TravelPlan failures

This replaces the earlier proxy-only analysis for the travel agent. The TravelPlanner section now uses actual execution-agent tool runs exported from LangSmith.

In [1]:
from __future__ import annotations

import json
import math
import re
from collections import Counter
from datetime import datetime
from itertools import combinations
from pathlib import Path
from urllib.parse import urlparse

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
from IPython.display import Markdown, display

ROOT = Path.cwd()
BASELINE_DIR = ROOT / "baseline"
TRAVEL_DIR = ROOT / "travel_agent"
THREAD_DIR = ROOT / "thread_analysis" / "travel_agent"

sns.set_theme(style="whitegrid", context="talk")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)

VALID_CATEGORIES = {"meal", "attraction", "transport", "lodging", "leisure", "other"}
SEARCH_TOOLS = {"search_web", "search_restaurants", "search_attractions", "search_flights", "search_hotels", "check_route_timing", "build_place_distance_graph"}
MUTATION_TOOLS = {"init_plan", "add_day", "remove_day", "add_slot", "insert_slot", "delete_slot", "view_plan", "cost_summary"}
FRAGILE_DOMAINS = re.compile(r"google\.com|maps\.google|instagram\.com|facebook\.com|tripadvisor\.com|booking\.com|expedia\.com|agoda\.com|turbopass\.com", re.I)
ERROR_RE = re.compile(r"timeout|timed out|error|failed|failure|exception|traceback|rate limit|unavailable|could not|unable|no reliable|missing info|missing_info", re.I)
UNCERTAIN_RE = re.compile(r"verify|re-verify|estimated|estimate|uncertain|unavailable|not available|backup|fallback|could not|depends|must be checked|not confirmed", re.I)

def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def safe_loads(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return {}
    if isinstance(value, dict):
        return value
    try:
        return json.loads(value)
    except Exception:
        return {"_raw": str(value)}

def safe_json(value) -> str:
    try:
        return json.dumps(value, ensure_ascii=False, default=str)
    except Exception:
        return str(value)

def canonical(obj) -> str:
    if obj is None or (isinstance(obj, float) and np.isnan(obj)):
        return ""
    if not isinstance(obj, str):
        obj = json.dumps(obj, ensure_ascii=False, sort_keys=True)
    obj = obj.lower()
    obj = re.sub(r"https?://\S+", " ", obj)
    obj = re.sub(r"[^\w\s€$£.-]", " ", obj)
    return re.sub(r"\s+", " ", obj).strip()

def arg_text(args: dict) -> str:
    if not isinstance(args, dict):
        return canonical(args)
    for key in ["query", "request", "text", "prompt", "origin", "destination", "place", "location"]:
        if key in args and args[key]:
            return canonical(args[key])
    return canonical(args)

def duration_seconds(start, end):
    try:
        return (pd.to_datetime(end, utc=True) - pd.to_datetime(start, utc=True)).total_seconds()
    except Exception:
        return np.nan

## 1. Load Real Tool Runs And Local Trace Metadata

In [2]:
manifest = pd.read_csv(THREAD_DIR / "manifest.csv")
travel_runs = pd.read_csv(THREAD_DIR / "langsmith_runs.csv")
travel_tools = pd.read_csv(THREAD_DIR / "tool_calls.csv")

travel_runs["duration_s"] = [duration_seconds(s, e) for s, e in zip(travel_runs.start_time, travel_runs.end_time)]
travel_runs["tool_args"] = travel_runs.inputs_json.map(safe_loads)
travel_runs["args_norm"] = travel_runs.tool_args.map(canonical)
travel_runs["query_norm"] = travel_runs.tool_args.map(arg_text)
travel_runs["output_text"] = travel_runs.outputs_json.fillna("").astype(str)
travel_runs["has_error_language"] = travel_runs.output_text.str.contains(ERROR_RE, na=False)
travel_runs["has_uncertainty_language"] = travel_runs.output_text.str.contains(UNCERTAIN_RE, na=False)
travel_runs = travel_runs.merge(manifest[["thread_id", "root_run_id", "query", "validation_passed", "validation_attempts", "travelplan_title"]], on="thread_id", how="left")
travel_runs = travel_runs.sort_values(["thread_id", "start_time", "id"]).reset_index(drop=True)
travel_runs["call_index"] = travel_runs.groupby("thread_id").cumcount()

display(Markdown(f"Loaded **{len(travel_runs):,} real TravelPlanner tool runs** across **{travel_runs.thread_id.nunique()} LangGraph threads**."))
display(travel_runs[["thread_id", "root_run_id", "name", "start_time", "duration_s", "query_norm", "validation_passed", "validation_attempts"]].head(20))

fig = px.bar(travel_runs.name.value_counts().reset_index(name="count"), x="name", y="count", title="Real TravelPlanner Tool Runs By Tool Name")
fig.update_layout(height=500, xaxis_tickangle=-35)
fig.show()

Loaded **2,300 real TravelPlanner tool runs** across **15 LangGraph threads**.

,thread_id,root_run_id,name,start_time,duration_s,query_norm,validation_passed,validation_attempts
0,0a464cd8-a08e-4d26-8b78-48de32de8a40,019e317e-c71c-73f0-a34e-3a3b57357b6c,write_todos,2026-05-16T15:54:22.577906,0.000599,todos content search budget flights and hostel options for munich cape town june 1 21 2026 status in_progress content verify key logistics airport transfers cape point transpor...,True,1
1,0a464cd8-a08e-4d26-8b78-48de32de8a40,019e317e-c71c-73f0-a34e-3a3b57357b6c,search_flights,2026-05-16T15:54:26.771101,4.510335,round-trip budget flight from munich muc to cape town cpt departing 2026-06-01 returning 2026-06-21 for 1 adult,True,1
2,0a464cd8-a08e-4d26-8b78-48de32de8a40,019e317e-c71c-73f0-a34e-3a3b57357b6c,search_hotels,2026-05-16T15:54:26.772240,19.313115,social budget hostel in cape town south africa from 2026-06-01 to 2026-06-21 for 1 adult solo traveler max budget 35 eur per night need wifi shared dorm good for meeting other ...,True,1
3,0a464cd8-a08e-4d26-8b78-48de32de8a40,019e317e-c71c-73f0-a34e-3a3b57357b6c,search_web,2026-05-16T15:54:26.772586,11.329306,official table mountain aerial cableway operating hours annual maintenance june 2026 safety guidance hiking solo cape town,True,1
4,0a464cd8-a08e-4d26-8b78-48de32de8a40,019e317e-c71c-73f0-a34e-3a3b57357b6c,search_web,2026-05-16T15:54:26.775387,12.794765,cape point cape of good hope official opening hours entry fee hiking wildlife june 2026 public transport from cape town,True,1
5,0a464cd8-a08e-4d26-8b78-48de32de8a40,019e317e-c71c-73f0-a34e-3a3b57357b6c,search_web,2026-05-16T15:54:26.776550,14.753983,budget wildlife safari day trip from cape town aquila or west coast national park prices solo traveler june 2026,True,1
6,0a464cd8-a08e-4d26-8b78-48de32de8a40,019e317e-c71c-73f0-a34e-3a3b57357b6c,write_todos,2026-05-16T15:54:49.099498,0.000602,todos content search budget flights and hostel options for munich cape town june 1 21 2026 status completed content verify key logistics airport transfers cape point transport ...,True,1
7,0a464cd8-a08e-4d26-8b78-48de32de8a40,019e317e-c71c-73f0-a34e-3a3b57357b6c,check_route_timing,2026-05-16T15:54:53.539953,0.001568,destination_address 81 long street cape town city centre cape town south africa origin_address cape town international airport cape town south africa travel_mode drive,True,1
8,0a464cd8-a08e-4d26-8b78-48de32de8a40,019e317e-c71c-73f0-a34e-3a3b57357b6c,check_route_timing,2026-05-16T15:54:53.540423,0.001512,destination_address muizenberg beach cape town south africa origin_address 81 long street cape town city centre cape town south africa travel_mode transit,True,1
9,0a464cd8-a08e-4d26-8b78-48de32de8a40,019e317e-c71c-73f0-a34e-3a3b57357b6c,check_route_timing,2026-05-16T15:54:53.540752,0.001518,destination_address v a waterfront cape town south africa origin_address 81 long street cape town city centre cape town south africa travel_mode walk,True,1


## 2. Baseline Tavily Tool Calls

In [3]:
baseline_rows = []
for path in sorted(BASELINE_DIR.glob("run-*.json")):
    obj = load_json(path)
    messages = (obj.get("outputs") or {}).get("messages") or []
    run_id = path.stem.removeprefix("run-")
    for mi, msg in enumerate(messages):
        for ci, call in enumerate(msg.get("tool_calls") or []):
            args = call.get("args") or {}
            baseline_rows.append({
                "system": "baseline",
                "thread_id": run_id,
                "root_run_id": run_id,
                "name": call.get("name"),
                "call_index": len(baseline_rows),
                "message_index": mi,
                "tool_args": args,
                "args_norm": canonical(args),
                "query_norm": arg_text(args),
                "duration_s": np.nan,
            })

baseline_tools = pd.DataFrame(baseline_rows)
display(Markdown(f"Loaded **{len(baseline_tools):,} explicit baseline Tavily tool calls** from local JSON exports."))
display(baseline_tools.head(20))

Loaded **193 explicit baseline Tavily tool calls** from local JSON exports.

,system,thread_id,root_run_id,name,call_index,message_index,tool_args,args_norm,query_norm,duration_s
0,baseline,019e31d9-cb29-70f1-87d9-a7166645dacc,019e31d9-cb29-70f1-87d9-a7166645dacc,tavily_web_search,0,2,{'query': 'official Munich to Vienna train ÖBB July 2026 fares travel time'},query official munich to vienna train öbb july 2026 fares travel time,official munich to vienna train öbb july 2026 fares travel time,NaN
1,baseline,019e31d9-cb29-70f1-87d9-a7166645dacc,019e31d9-cb29-70f1-87d9-a7166645dacc,tavily_web_search,1,2,{'query': 'Vienna public transport tickets 24 48 72 hour pass official Wiener Linien prices'},query vienna public transport tickets 24 48 72 hour pass official wiener linien prices,vienna public transport tickets 24 48 72 hour pass official wiener linien prices,NaN
2,baseline,019e31d9-cb29-70f1-87d9-a7166645dacc,019e31d9-cb29-70f1-87d9-a7166645dacc,tavily_web_search,2,2,{'query': 'Vienna Kunsthistorisches Museum official opening hours tickets 2026'},query vienna kunsthistorisches museum official opening hours tickets 2026,vienna kunsthistorisches museum official opening hours tickets 2026,NaN
3,baseline,019e31d9-cb29-70f1-87d9-a7166645dacc,019e31d9-cb29-70f1-87d9-a7166645dacc,tavily_web_search,3,2,{'query': 'Vienna Belvedere Museum official tickets opening hours 2026'},query vienna belvedere museum official tickets opening hours 2026,vienna belvedere museum official tickets opening hours 2026,NaN
4,baseline,019e31d9-cb29-70f1-87d9-a7166645dacc,019e31d9-cb29-70f1-87d9-a7166645dacc,tavily_web_search,4,2,{'query': 'Vienna hotel budget double room July 2026 official Motel One ibis B&B hotel'},query vienna hotel budget double room july 2026 official motel one ibis b b hotel,vienna hotel budget double room july 2026 official motel one ibis b b hotel,NaN
5,baseline,019e31d9-cb29-70f1-87d9-a7166645dacc,019e31d9-cb29-70f1-87d9-a7166645dacc,tavily_web_search,5,8,{'query': 'official Vienna tourism walking tours Ringstrasse historic center city walks'},query official vienna tourism walking tours ringstrasse historic center city walks,official vienna tourism walking tours ringstrasse historic center city walks,NaN
6,baseline,019e31d9-cb29-70f1-87d9-a7166645dacc,019e31d9-cb29-70f1-87d9-a7166645dacc,tavily_web_search,6,8,{'query': 'official Albertina Vienna opening hours tickets 2026'},query official albertina vienna opening hours tickets 2026,official albertina vienna opening hours tickets 2026,NaN
7,baseline,019e31d9-cb29-70f1-87d9-a7166645dacc,019e31d9-cb29-70f1-87d9-a7166645dacc,tavily_web_search,7,8,{'query': 'official Schönbrunn Palace Vienna opening hours tickets 2026'},query official schönbrunn palace vienna opening hours tickets 2026,official schönbrunn palace vienna opening hours tickets 2026,NaN
8,baseline,019e31d9-cb29-70f1-87d9-a7166645dacc,019e31d9-cb29-70f1-87d9-a7166645dacc,tavily_web_search,8,8,{'query': 'Vienna traditional restaurants official Figlmuller Plachutta Naschmarkt opening hours'},query vienna traditional restaurants official figlmuller plachutta naschmarkt opening hours,vienna traditional restaurants official figlmuller plachutta naschmarkt opening hours,NaN
9,baseline,019e31d9-cb29-70f1-87d9-a7166645dacc,019e31d9-cb29-70f1-87d9-a7166645dacc,tavily_web_search,9,8,{'query': 'Vienna budget hotels official Motel One Wien Hauptbahnhof B&B Hotel Wien Hbf July 2026 rates'},query vienna budget hotels official motel one wien hauptbahnhof b b hotel wien hbf july 2026 rates,vienna budget hotels official motel one wien hauptbahnhof b b hotel wien hbf july 2026 rates,NaN


## 3. Repeated Tool Calls

In [4]:
travel_dupes = (
    travel_runs.groupby(["thread_id", "name", "args_norm"])
    .agg(count=("id", "size"), first_call=("call_index", "min"), last_call=("call_index", "max"), example_args=("tool_args", "first"), root_run_id=("root_run_id", "first"))
    .reset_index()
    .query("count > 1")
    .sort_values("count", ascending=False)
)
baseline_dupes = (
    baseline_tools.groupby(["thread_id", "name", "args_norm"])
    .agg(count=("name", "size"), first_call=("call_index", "min"), last_call=("call_index", "max"), example_args=("tool_args", "first"), root_run_id=("root_run_id", "first"))
    .reset_index()
    .query("count > 1")
    .sort_values("count", ascending=False)
)

display(Markdown("### TravelPlanner repeated exact tool calls"))
display(travel_dupes.head(50))
display(Markdown("### Baseline repeated exact Tavily calls"))
display(baseline_dupes.head(50))

rep_summary = pd.concat([
    travel_dupes.assign(system="travel_agent"),
    baseline_dupes.assign(system="baseline"),
], ignore_index=True)
if not rep_summary.empty:
    fig = px.bar(rep_summary.groupby(["system", "name"]).agg(repeated_groups=("args_norm", "size"), repeated_calls=("count", "sum")).reset_index(),
                 x="name", y="repeated_calls", color="system", barmode="group", title="Repeated Exact Tool Calls")
    fig.update_layout(height=550, xaxis_tickangle=-35)
    fig.show()

### TravelPlanner repeated exact tool calls

,thread_id,name,args_norm,count,first_call,last_call,example_args,root_run_id
1860,c78a54be-a5c4-4f1f-924c-fb9eb3b00392,view_plan,,29,17,534,{},019e30f5-baf1-73e3-9d80-fac667a9d285
1741,c78a54be-a5c4-4f1f-924c-fb9eb3b00392,delete_slot,day_index 3 position 2,15,347,419,"{'day_index': 3, 'position': 2}",019e30f5-baf1-73e3-9d80-fac667a9d285
1729,c78a54be-a5c4-4f1f-924c-fb9eb3b00392,cost_summary,,14,180,533,{},019e30f5-baf1-73e3-9d80-fac667a9d285
893,27d9e1ea-bbf0-4faf-99bf-3a96c9d58cad,view_plan,,10,99,196,{},019e3140-2290-7743-b923-de2df567958e
1991,cdf0dca5-6c98-4138-8f47-f3011dfa953a,view_plan,,8,51,123,{},019e3160-f2d0-71e2-9549-5b51d9dd2a4b
1742,c78a54be-a5c4-4f1f-924c-fb9eb3b00392,delete_slot,day_index 3 position 3,8,124,482,"{'day_index': 3, 'position': 3}",019e30f5-baf1-73e3-9d80-fac667a9d285
799,27d9e1ea-bbf0-4faf-99bf-3a96c9d58cad,cost_summary,,8,121,208,{},019e3140-2290-7743-b923-de2df567958e
1779,c78a54be-a5c4-4f1f-924c-fb9eb3b00392,insert_slot,category meal cost 30 day_index 3 description budget lunch onigiri bento gyudon or supermarket meal. end_time_iso 2026-09-22t13 15 links location hostel area name lunch konbini...,6,351,394,"{'category': 'meal', 'cost': 30, 'day_index': 3, 'description': 'Budget lunch: onigiri, bento, gyudon, or supermarket meal.', 'end_time_iso': '2026-09-22T13:15', 'links': [], '...",019e30f5-baf1-73e3-9d80-fac667a9d285
1471,aeb27ff7-ab7d-4164-aca5-6633c26c24d6,view_plan,,6,42,100,{},019e316b-92a9-7710-8b5f-68fffa29eed2
1768,c78a54be-a5c4-4f1f-924c-fb9eb3b00392,insert_slot,category leisure cost 0 day_index 3 description rest after overnight travel set up ic cards cash and ticket accounts in the common area. end_time_iso 2026-09-22t12 00 links loc...,6,350,393,"{'category': 'leisure', 'cost': 0, 'day_index': 3, 'description': 'Rest after overnight travel; set up IC cards, cash, and ticket accounts in the common area.', 'end_time_iso':...",019e30f5-baf1-73e3-9d80-fac667a9d285


### Baseline repeated exact Tavily calls

,thread_id,name,args_norm,count,first_call,last_call,example_args,root_run_id


## 4. Repeated Queries

In [5]:
travel_search = travel_runs[travel_runs.name.isin(SEARCH_TOOLS)].copy()
baseline_search = baseline_tools.copy()

travel_query_dupes = (
    travel_search.groupby(["thread_id", "name", "query_norm"])
    .agg(count=("id", "size"), example_args=("tool_args", "first"), root_run_id=("root_run_id", "first"))
    .reset_index()
    .query("query_norm != '' and count > 1")
    .sort_values("count", ascending=False)
)
baseline_query_dupes = (
    baseline_search.groupby(["thread_id", "name", "query_norm"])
    .agg(count=("name", "size"), example_args=("tool_args", "first"), root_run_id=("root_run_id", "first"))
    .reset_index()
    .query("query_norm != '' and count > 1")
    .sort_values("count", ascending=False)
)

display(Markdown("### TravelPlanner repeated search/subagent queries"))
display(travel_query_dupes.head(50))
display(Markdown("### Baseline repeated Tavily queries"))
display(baseline_query_dupes.head(50))

qsum = pd.concat([travel_query_dupes.assign(system="travel_agent"), baseline_query_dupes.assign(system="baseline")], ignore_index=True)
if not qsum.empty:
    fig = px.histogram(qsum, x="count", color="system", marginal="box", title="Distribution Of Repeated Query Counts")
    fig.show()

### TravelPlanner repeated search/subagent queries

,thread_id,name,query_norm,count,example_args,root_run_id


### Baseline repeated Tavily queries

,thread_id,name,query_norm,count,example_args,root_run_id


## 5. Dead Loops

In [6]:
loop_rows = []

def detect_loops(df, system):
    for thread_id, g in df.sort_values(["thread_id", "call_index"]).groupby("thread_id"):
        rows = g.to_dict("records")
        # Consecutive exact repeat: A, A, A
        streak = 1
        prev_key = None
        for row in rows:
            key = (row["name"], row.get("args_norm") or row.get("query_norm"))
            if key == prev_key:
                streak += 1
            else:
                streak = 1
            if streak >= 2:
                loop_rows.append({"system": system, "thread_id": thread_id, "root_run_id": row.get("root_run_id"), "loop_type": "consecutive_exact_repeat", "tool": row["name"], "count": streak, "example": row.get("query_norm") or row.get("args_norm")})
            prev_key = key

        # Alternating mutation loop: add/delete/add/delete or insert/delete repeated on same day/position-ish args.
        for i in range(len(rows) - 3):
            seq = rows[i:i+4]
            names = [r["name"] for r in seq]
            if names in (["add_slot", "delete_slot", "add_slot", "delete_slot"], ["insert_slot", "delete_slot", "insert_slot", "delete_slot"], ["delete_slot", "add_slot", "delete_slot", "add_slot"], ["delete_slot", "insert_slot", "delete_slot", "insert_slot"]):
                loop_rows.append({"system": system, "thread_id": thread_id, "root_run_id": seq[-1].get("root_run_id"), "loop_type": "mutation_backtrack_loop", "tool": " -> ".join(names), "count": 4, "example": safe_json([s.get("tool_args") for s in seq])})

        # Excessive same query in one thread.
        counts = Counter((r["name"], r.get("query_norm") or r.get("args_norm")) for r in rows)
        for (tool, q), count in counts.items():
            if q and count >= 3:
                loop_rows.append({"system": system, "thread_id": thread_id, "root_run_id": rows[0].get("root_run_id"), "loop_type": "same_call_3plus", "tool": tool, "count": count, "example": q})

detect_loops(travel_runs, "travel_agent")
detect_loops(baseline_tools, "baseline")
loops_df = pd.DataFrame(loop_rows).drop_duplicates() if loop_rows else pd.DataFrame()

display(loops_df.sort_values(["system", "count"], ascending=[True, False]).head(100) if not loops_df.empty else pd.DataFrame({"status": ["No dead-loop indicators found"]}))
if not loops_df.empty:
    fig = px.bar(loops_df.groupby(["system", "loop_type"]).size().reset_index(name="events"), x="loop_type", y="events", color="system", barmode="group", title="Dead-Loop Indicators")
    fig.update_layout(height=500, xaxis_tickangle=-25)
    fig.show()

,system,thread_id,root_run_id,loop_type,tool,count,example
25,travel_agent,1cafa93c-cb33-4393-a210-c1527aedcdb0,019e3150-02bf-7471-8e63-f12ad2e4bcb4,same_call_3plus,add_slot,37,noo faru kagi maldives resort spa
34,travel_agent,1cafa93c-cb33-4393-a210-c1527aedcdb0,019e3150-02bf-7471-8e63-f12ad2e4bcb4,same_call_3plus,add_slot,15,oceans restaurant centara ras fushi
46,travel_agent,26f475de-7862-44f9-b5c4-876d50bc6ed7,019e319b-12cf-77f1-ba10-95817e5a4aa5,same_call_3plus,add_slot,15,villa
127,travel_agent,c78a54be-a5c4-4f1f-924c-fb9eb3b00392,019e30f5-baf1-73e3-9d80-fac667a9d285,same_call_3plus,delete_slot,15,day_index 3 position 2
30,travel_agent,1cafa93c-cb33-4393-a210-c1527aedcdb0,019e3150-02bf-7471-8e63-f12ad2e4bcb4,same_call_3plus,add_slot,12,kagi maldives resort spa
...,...,...,...,...,...,...,...
38,travel_agent,1df2a856-1e25-4921-a93e-a02c1c1257a9,019e30f0-c457-7b72-9ad9-01656f704cdc,same_call_3plus,add_slot,3,midtown manhattan client offices
57,travel_agent,27d9e1ea-bbf0-4faf-99bf-3a96c9d58cad,019e3140-2290-7743-b923-de2df567958e,same_call_3plus,add_slot,3,riad tamarrakecht bab doukkala
60,travel_agent,27d9e1ea-bbf0-4faf-99bf-3a96c9d58cad,019e3140-2290-7743-b923-de2df567958e,same_call_3plus,insert_slot,3,riad tamarrakecht bab aylen medina
61,travel_agent,27d9e1ea-bbf0-4faf-99bf-3a96c9d58cad,019e3140-2290-7743-b923-de2df567958e,same_call_3plus,delete_slot,3,day_index 6 position 7


## 6. Timeouts And Long-Running Tool Calls

In [7]:
# No explicit tool errors were returned in the exported tool runs, so timeout risk is approximated by latency outliers and error language.
p95 = travel_runs.duration_s.quantile(0.95)
p99 = travel_runs.duration_s.quantile(0.99)
timeout_df = travel_runs[(travel_runs.duration_s >= p99) | travel_runs.has_error_language].copy()
timeout_df["reason"] = np.where(timeout_df.has_error_language, "error_language", "p99_latency_outlier")

display(Markdown(f"Tool latency thresholds: p95={p95:.2f}s, p99={p99:.2f}s."))
display(timeout_df[["thread_id", "root_run_id", "name", "duration_s", "reason", "query_norm", "output_text"]].sort_values("duration_s", ascending=False).head(100))

fig = px.box(travel_runs, x="name", y="duration_s", points=False, title="TravelPlanner Tool Run Latency By Tool")
fig.update_layout(height=550, xaxis_tickangle=-35)
fig.show()

Tool latency thresholds: p95=14.22s, p99=22.24s.

,thread_id,root_run_id,name,duration_s,reason,query_norm,output_text
1626,c78a54be-a5c4-4f1f-924c-fb9eb3b00392,019e30f5-baf1-73e3-9d80-fac667a9d285,search_flights,51.908851,p99_latency_outlier,round-trip flights for 6 adults from munich muc to tokyo hnd or nrt departing 2026-09-20 and returning 2026-09-30 economy class budget group trip,"{""output"": {""additional_kwargs"": {}, ""content"": ""round trip | MUC → HND on 2026-09-20 (return 2026-09-30) | adults: 6 | currency: EUR\nStatus: success\nSelected flights (2):\n ..."
1529,aeb27ff7-ab7d-4164-aca5-6633c26c24d6,019e316b-92a9-7710-8b5f-68fffa29eed2,search_attractions,51.476030,p99_latency_outlier,find a guided astronomical observatory or roque de los muchachos experience for one solo traveller on la palma canary islands on day 4 of a nov 2026 astronomy trip budget 80 eu...,"{""output"": {""additional_kwargs"": {}, ""content"": ""Destination: La Palma Canary Islands | Budget: 80.0 EUR | Archetype: active_solo_traveler | Status: success\nActivity — Day 4 A..."
1472,ab1c48a7-8d2b-49ee-9868-5097cebf14f8,019e3139-9620-78e2-9742-b53189867ea9,search_web,38.104027,p99_latency_outlier,official vienna airport restaurants terminal lunch dinner opening hours june 2026,"{""output"": {""additional_kwargs"": {}, ""content"": ""**Wolfgang Puck Kitchen + Bar**\n- **Location/Transit:** Terminal 3, Level 2 (Arrivals Hall). Accessible to non-passengers. Nea..."
1524,aeb27ff7-ab7d-4164-aca5-6633c26c24d6,019e316b-92a9-7710-8b5f-68fffa29eed2,search_flights,36.424873,p99_latency_outlier,round-trip flight for 1 adult from hamburg ham to la palma spc departing 2026-11-12 and returning 2026-11-19 economy,"{""output"": {""additional_kwargs"": {}, ""content"": ""round trip | HAM → SPC on 2026-11-12 (return 2026-11-19) | adults: 1 | currency: EUR\nStatus: success\nSelected flights (0):\nP..."
669,1df2a856-1e25-4921-a93e-a02c1c1257a9,019e30f0-c457-7b72-9ad9-01656f704cdc,search_web,32.342864,error_language,condor frankfurt to new york jfk de2016 terminal frankfurt jfk june 2026,"{""output"": {""additional_kwargs"": {}, ""content"": ""**Condor Flight DE2016 (Frankfurt to New York JFK)**\n- **Location/Transit:** Departs Frankfurt Airport (FRA) Terminal 1; arriv..."
...,...,...,...,...,...,...,...
380,1cafa93c-cb33-4393-a210-c1527aedcdb0,019e3150-02bf-7471-8e63-f12ad2e4bcb4,search_hotels,1.877669,error_language,luxury overwater bungalow or high-end beach resort in maldives from 2026-10-10 to 2026-10-20 for 2 adults budget around 500 eur per night or best value within total trip budget...,"{""output"": {""additional_kwargs"": {}, ""content"": ""Hotel search | Maldives | 2026-10-10 to 2026-10-20 (10 nights)\nGuests: 2 | Budget: 800.0 EUR/night | Status: failed\n Error [..."
1689,c78a54be-a5c4-4f1f-924c-fb9eb3b00392,019e30f5-baf1-73e3-9d80-fac667a9d285,search_restaurants,1.717138,error_language,kichijoji tokyo cheap lunch for 6 people near inokashira park or ghibli museum japanese casual,"{""output"": {""additional_kwargs"": {}, ""content"": ""Restaurant search | Tokyo\nQuery: Kichijoji Tokyo cheap lunch for 6 people near Inokashira Park or Ghibli Museum Japanese casua..."
1542,aeb27ff7-ab7d-4164-aca5-6633c26c24d6,019e316b-92a9-7710-8b5f-68fffa29eed2,check_route_timing,0.005027,error_language,destination_address restaurante jardin de los naranjos camino el pinar 33 38789 puntagorda la palma spain origin_address pension roque faro calle roque faro 38788 garafía la pa...,"{""output"": {""additional_kwargs"": {}, ""content"": ""{\""ok\"": false, \""stage\"": \""api_key\"", \""error\"": \""Missing Google Maps API key\""}"", ""name"": ""check_route_timing"", ""response_m..."
2193,cdf0dca5-6c98-4138-8f47-f3011dfa953a,019e3160-f2d0-71e2-9549-5b51d9dd2a4b,check_route_timing,0.004244,error_language,destination_address bonboon amsterdam netherlands origin_address conscious hotel westerpark haarlemmerweg 10 1014 be amsterdam netherlands travel_mode bicycling,"{""output"": {""additional_kwargs"": {}, ""content"": ""{\""ok\""

## 7. Cascading Errors

In [8]:
cascade_rows = []
for thread_id, g in travel_runs.sort_values(["thread_id", "call_index"]).groupby("thread_id"):
    meta = g.iloc[0]
    if meta.validation_attempts and meta.validation_attempts > 1:
        cascade_rows.append({"thread_id": thread_id, "root_run_id": meta.root_run_id, "cascade_type": "validator_repair_loop", "severity": "high", "detail": f"validation_attempts={meta.validation_attempts}"})

    # Search/tool uncertainty followed by plan mutation means weak evidence entered the itinerary-building phase.
    uncertain_indices = set(g[g.has_uncertainty_language | g.has_error_language].call_index)
    for idx in uncertain_indices:
        later = g[(g.call_index > idx) & (g.name.isin({"add_slot", "insert_slot"}))].head(3)
        if not later.empty:
            src = g[g.call_index == idx].iloc[0]
            cascade_rows.append({"thread_id": thread_id, "root_run_id": meta.root_run_id, "cascade_type": "weak_evidence_then_plan_mutation", "severity": "medium", "detail": f"{src.name} at call {idx} followed by {', '.join(later.name.tolist())}"})

    # Delete after add/insert is a repair/backtracking signal.
    for i, row in g[g.name == "delete_slot"].iterrows():
        prev_mut = g[(g.call_index < row.call_index) & (g.name.isin({"add_slot", "insert_slot"}))].tail(1)
        if not prev_mut.empty:
            cascade_rows.append({"thread_id": thread_id, "root_run_id": meta.root_run_id, "cascade_type": "mutation_repair_delete", "severity": "medium", "detail": f"delete_slot after {prev_mut.iloc[0].name}"})

cascade_df = pd.DataFrame(cascade_rows).drop_duplicates() if cascade_rows else pd.DataFrame()
display(cascade_df.head(100) if not cascade_df.empty else pd.DataFrame({"status": ["No cascade indicators found"]}))
if not cascade_df.empty:
    fig = px.bar(cascade_df.groupby(["cascade_type", "severity"]).size().reset_index(name="events"), x="cascade_type", y="events", color="severity", title="Cascading Error Indicators")
    fig.update_layout(height=500, xaxis_tickangle=-30)
    fig.show()

,thread_id,root_run_id,cascade_type,severity,detail
0,0a464cd8-a08e-4d26-8b78-48de32de8a40,019e317e-c71c-73f0-a34e-3a3b57357b6c,weak_evidence_then_plan_mutation,medium,"0 at call 0 followed by add_slot, add_slot, add_slot"
1,0a464cd8-a08e-4d26-8b78-48de32de8a40,019e317e-c71c-73f0-a34e-3a3b57357b6c,weak_evidence_then_plan_mutation,medium,"1 at call 1 followed by add_slot, add_slot, add_slot"
2,0a464cd8-a08e-4d26-8b78-48de32de8a40,019e317e-c71c-73f0-a34e-3a3b57357b6c,weak_evidence_then_plan_mutation,medium,"3 at call 3 followed by add_slot, add_slot, add_slot"
3,0a464cd8-a08e-4d26-8b78-48de32de8a40,019e317e-c71c-73f0-a34e-3a3b57357b6c,weak_evidence_then_plan_mutation,medium,"6 at call 6 followed by add_slot, add_slot, add_slot"
4,0a464cd8-a08e-4d26-8b78-48de32de8a40,019e317e-c71c-73f0-a34e-3a3b57357b6c,weak_evidence_then_plan_mutation,medium,"7 at call 7 followed by add_slot, add_slot, add_slot"
...,...,...,...,...,...
95,1df2a856-1e25-4921-a93e-a02c1c1257a9,019e30f0-c457-7b72-9ad9-01656f704cdc,weak_evidence_then_plan_mutation,medium,"639 at call 3 followed by add_slot, add_slot, add_slot"
96,1df2a856-1e25-4921-a93e-a02c1c1257a9,019e30f0-c457-7b72-9ad9-01656f704cdc,weak_evidence_then_plan_mutation,medium,"640 at call 4 followed by add_slot, add_slot, add_slot"
97,1df2a856-1e25-4921-a93e-a02c1c1257a9,019e30f0-c457-7b72-9ad9-01656f704cdc,weak_evidence_then_plan_mutation,medium,"647 at call 11 followed by add_slot, add_slot, add_slot"
98,1df2a856-1e25-4921-a93e-a02c1c1257a9,019e30f0-c457-7b72-9ad9-01656f704cdc,weak_evidence_then_plan_mutation,medium,"652 at call 16 followed by add_slot, add_slot, add_slot"


## 8. Structural TravelPlan Validation

In [9]:
structural_rows = []
slot_rows = []
for path in sorted(TRAVEL_DIR.glob("run-*.json")):
    obj = load_json(path)
    outputs = obj.get("outputs") or {}
    metadata = obj.get("metadata") or {}
    thread_id = metadata.get("thread_id")
    root_run_id = path.stem.removeprefix("run-")
    budget = (outputs.get("normalized_constraints") or {}).get("budget_amount")
    plan = outputs.get("travelplan") or {}

    total_cost = 0.0
    for day in plan.get("days") or []:
        slots = day.get("slots") or []
        parsed_slots = []
        for pos, slot in enumerate(slots, start=1):
            start = pd.to_datetime(slot.get("start_time"), errors="coerce")
            end = pd.to_datetime(slot.get("end_time"), errors="coerce")
            links = slot.get("links") or []
            cost = slot.get("cost") or 0
            total_cost += cost
            text = " ".join(str(slot.get(k) or "") for k in ["name", "description", "location", "notes"])
            row = {
                "thread_id": thread_id,
                "root_run_id": root_run_id,
                "day_index": day.get("index"),
                "position": pos,
                "slot_name": slot.get("name"),
                "category": slot.get("category"),
                "start_time": start,
                "end_time": end,
                "cost": cost,
                "links": links,
                "missing_links": len(links) == 0,
                "fragile_links": any(FRAGILE_DOMAINS.search(link or "") for link in links),
                "unknown_category": slot.get("category") not in VALID_CATEGORIES,
                "invalid_time": pd.isna(start) or pd.isna(end) or end <= start,
                "uncertainty_note": bool(UNCERTAIN_RE.search(text)),
            }
            slot_rows.append(row)
            parsed_slots.append(row)

        for a, b in combinations(parsed_slots, 2):
            if not a["invalid_time"] and not b["invalid_time"] and a["start_time"] < b["end_time"] and b["start_time"] < a["end_time"]:
                structural_rows.append({"thread_id": thread_id, "root_run_id": root_run_id, "check": "overlapping_slots", "detail": f"Day {day.get('index')}: {a['slot_name']} overlaps {b['slot_name']}"})

    if budget is not None and total_cost > float(budget):
        structural_rows.append({"thread_id": thread_id, "root_run_id": root_run_id, "check": "budget_exceeded", "detail": f"computed_total={total_cost:.2f}, budget={budget}"})

slots_df = pd.DataFrame(slot_rows)
for check_col in ["missing_links", "fragile_links", "unknown_category", "invalid_time", "uncertainty_note"]:
    bad = slots_df[slots_df[check_col]]
    for _, row in bad.iterrows():
        structural_rows.append({"thread_id": row.thread_id, "root_run_id": row.root_run_id, "check": check_col, "detail": f"Day {row.day_index} slot {row.position}: {row.slot_name}"})

structural_df = pd.DataFrame(structural_rows)
display(structural_df.head(100) if not structural_df.empty else pd.DataFrame({"status": ["No structural TravelPlan failures found"]}))

structural_summary = structural_df.groupby("check").size().reset_index(name="events") if not structural_df.empty else pd.DataFrame(columns=["check", "events"])
display(structural_summary)
if not structural_summary.empty:
    fig = px.bar(structural_summary, x="check", y="events", title="Structural TravelPlan Error Checks")
    fig.update_layout(height=500, xaxis_tickangle=-30)
    fig.show()

,thread_id,root_run_id,check,detail
0,1665af40-4b03-45b0-9de8-d03aac32994f,019e30e2-040b-79d2-b82e-a5ca9c720ad1,missing_links,Day 2 slot 1: Breakfast: local bakery/café near Neubau
1,1665af40-4b03-45b0-9de8-d03aac32994f,019e30e2-040b-79d2-b82e-a5ca9c720ad1,missing_links,Day 4 slot 1: Breakfast: local bakery/café near Neubau
2,75d18508-a58a-4b3d-b6c3-8f98343efd9f,019e30e9-5a48-7d21-8141-76c25d06e1d0,missing_links,Day 1 slot 2: Early dinner at Berlin airport
3,75d18508-a58a-4b3d-b6c3-8f98343efd9f,019e30e9-5a48-7d21-8141-76c25d06e1d0,missing_links,Day 2 slot 2: Sleep in and apartment breakfast
4,75d18508-a58a-4b3d-b6c3-8f98343efd9f,019e30e9-5a48-7d21-8141-76c25d06e1d0,missing_links,Day 2 slot 5: Rest and siesta at hotel
...,...,...,...,...
95,26f475de-7862-44f9-b5c4-876d50bc6ed7,019e319b-12cf-77f1-ba10-95817e5a4aa5,missing_links,Day 4 slot 1: Breakfast at villa
96,26f475de-7862-44f9-b5c4-876d50bc6ed7,019e319b-12cf-77f1-ba10-95817e5a4aa5,missing_links,Day 4 slot 5: Villa pool / rest afternoon
97,26f475de-7862-44f9-b5c4-876d50bc6ed7,019e319b-12cf-77f1-ba10-95817e5a4aa5,missing_links,Day 4 slot 6: Early family dinner at villa
98,26f475de-7862-44f9-b5c4-876d50bc6ed7,019e319b-12cf-77f1-ba10-95817e5a4aa5,missing_links,Day 5 slot 1: Early breakfast at villa


,check,events
0,fragile_links,134
1,missing_links,110
2,uncertainty_note,160


## 9. Paper Taxonomy Table

In [10]:
taxonomy_rows = []
taxonomy_rows.append({"category": "T2 Missing Mandatory Parameters", "baseline": int(len(baseline_query_dupes)), "travel_agent": int(len(travel_query_dupes)), "operationalization": "Repeated or underspecified query strings in real tool args; manual review examples above."})
taxonomy_rows.append({"category": "T3 Redundant / Infinite Loops", "baseline": int((loops_df.system == 'baseline').sum()) if not loops_df.empty else 0, "travel_agent": int((loops_df.system == 'travel_agent').sum()) if not loops_df.empty else 0, "operationalization": "Consecutive exact repeats, same call >=3 times, mutation add/delete loops."})
taxonomy_rows.append({"category": "T4 Misinterpreting Tool Outputs", "baseline": 0, "travel_agent": int(travel_runs.has_uncertainty_language.sum()), "operationalization": "Tool outputs containing uncertainty/fallback language that still feeds later plan mutations."})
taxonomy_rows.append({"category": "T5 Link Rot / Empty Evidence", "baseline": 0, "travel_agent": int(slots_df.missing_links.sum() + slots_df.fragile_links.sum()), "operationalization": "Final TravelPlan slots with no links or fragile domains."})
taxonomy_rows.append({"category": "T6 Cross-Agent Data Corruption", "baseline": 0, "travel_agent": int(len(cascade_df)) if not cascade_df.empty else 0, "operationalization": "Validator repair loops, delete-after-add repair, weak-evidence-then-mutation cascades."})
taxonomy = pd.DataFrame(taxonomy_rows)
display(taxonomy)

latex = taxonomy.to_latex(index=False, escape=True)
display(Markdown("### LaTeX table for `paper.tex`"))
print(latex)

,category,baseline,travel_agent,operationalization
0,T2 Missing Mandatory Parameters,0,0,Repeated or underspecified query strings in real tool args; manual review examples above.
1,T3 Redundant / Infinite Loops,0,132,"Consecutive exact repeats, same call >=3 times, mutation add/delete loops."
2,T4 Misinterpreting Tool Outputs,0,378,Tool outputs containing uncertainty/fallback language that still feeds later plan mutations.
3,T5 Link Rot / Empty Evidence,0,244,Final TravelPlan slots with no links or fragile domains.
4,T6 Cross-Agent Data Corruption,0,557,"Validator repair loops, delete-after-add repair, weak-evidence-then-mutation cascades."


### LaTeX table for `paper.tex`

\begin{tabular}{lrrl}
\toprule
category & baseline & travel\_agent & operationalization \\
\midrule
T2 Missing Mandatory Parameters & 0 & 0 & Repeated or underspecified query strings in real tool args; manual review examples above. \\
T3 Redundant / Infinite Loops & 0 & 132 & Consecutive exact repeats, same call >=3 times, mutation add/delete loops. \\
T4 Misinterpreting Tool Outputs & 0 & 378 & Tool outputs containing uncertainty/fallback language that still feeds later plan mutations. \\
T5 Link Rot / Empty Evidence & 0 & 244 & Final TravelPlan slots with no links or fragile domains. \\
T6 Cross-Agent Data Corruption & 0 & 557 & Validator repair loops, delete-after-add repair, weak-evidence-then-mutation cascades. \\
\bottomrule
\end{tabular}



## 10. Report-Ready Summary

In [11]:
summary = f'''
### Error Analysis Summary

The corrected analysis uses {len(travel_runs):,} real LangSmith tool runs from {travel_runs.thread_id.nunique()} TravelPlanner threads, rather than final-plan link proxies. The multi-agent system made heavy use of TravelPlan mutation tools (`add_slot`, `add_day`, `delete_slot`, `insert_slot`) and domain tools (`search_web`, `search_restaurants`, `search_attractions`, `search_flights`, `search_hotels`, `check_route_timing`).

Key detected indicators:

- Repeated exact TravelPlanner tool-call groups: {len(travel_dupes):,}
- Repeated TravelPlanner query groups: {len(travel_query_dupes):,}
- TravelPlanner dead-loop indicators: {int((loops_df.system == 'travel_agent').sum()) if not loops_df.empty else 0:,}
- Baseline dead-loop indicators: {int((loops_df.system == 'baseline').sum()) if not loops_df.empty else 0:,}
- TravelPlanner latency/error-language timeout indicators: {len(timeout_df):,}
- TravelPlanner cascade indicators: {len(cascade_df) if not cascade_df.empty else 0:,}
- Structural TravelPlan validation events: {len(structural_df) if not structural_df.empty else 0:,}

Interpretation for the paper: the baseline's failure mode is mostly repeated or underspecified Tavily search. The TravelPlanner's failure mode is more architectural: repeated plan mutations, repair loops, weak-evidence propagation, and fragile final evidence links. This supports the paper's claim that the multi-agent system improves organization and constraint handling but introduces cross-agent propagation risks.
'''
display(Markdown(summary))


### Error Analysis Summary

The corrected analysis uses 2,300 real LangSmith tool runs from 15 TravelPlanner threads, rather than final-plan link proxies. The multi-agent system made heavy use of TravelPlan mutation tools (`add_slot`, `add_day`, `delete_slot`, `insert_slot`) and domain tools (`search_web`, `search_restaurants`, `search_attractions`, `search_flights`, `search_hotels`, `check_route_timing`).

Key detected indicators:

- Repeated exact TravelPlanner tool-call groups: 131
- Repeated TravelPlanner query groups: 0
- TravelPlanner dead-loop indicators: 132
- Baseline dead-loop indicators: 0
- TravelPlanner latency/error-language timeout indicators: 210
- TravelPlanner cascade indicators: 557
- Structural TravelPlan validation events: 404

Interpretation for the paper: the baseline's failure mode is mostly repeated or underspecified Tavily search. The TravelPlanner's failure mode is more architectural: repeated plan mutations, repair loops, weak-evidence propagation, and fragile final evidence links. This supports the paper's claim that the multi-agent system improves organization and constraint handling but introduces cross-agent propagation risks.
